In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS olist.dwh")

DataFrame[]

In [0]:
customers = spark.table("olist.stg.customers")

window = Window.orderBy("customer_id")

dim_customers = (
    customers
    .select(
        row_number().over(window).alias("customer_sk"),
        col("customer_id"),
        col("customer_city"),
        col("customer_state"),
        col("customer_zip_code_prefix")
    )
)

dim_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.dwh.dim_customers")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
sellers = spark.table("olist.stg.sellers")

window = Window.orderBy("seller_id")

dim_sellers = (
    sellers
    .select(
        row_number().over(window).alias("seller_sk"),
        col("seller_id"),
        col("seller_city"),
        col("seller_state"),
        col("seller_zip_code_prefix")
    )
)

dim_sellers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.dwh.dim_sellers")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
products = spark.table("olist.stg.products")
translations = spark.table("olist.stg.category_translation")

window = Window.orderBy("product_id")

dim_products = (
    products.alias("p")
    .join(
        translations.alias("t"),
        col("p.product_category_name") == col("t.product_category_name"),
        "left"
    )
    .select(
        row_number().over(window).alias("product_sk"),

        col("p.product_id"),

        col("p.product_category_name"),

        coalesce(
            col("t.product_category_name_english"),
            lit("unknown")
        ).alias("product_category_name_english"),

        col("p.product_name_length"),

        col("p.product_description_length"),

        col("p.product_photos_qty"),

        col("p.product_weight_g"),

        col("p.product_length_cm"),

        col("p.product_height_cm"),

        col("p.product_width_cm")
    )
)

dim_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.dwh.dim_products")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
orders = spark.table("olist.stg.orders")
items = spark.table("olist.stg.order_items")
reviews = spark.table("olist.stg.order_reviews")

In [0]:
dates = (
    orders.select(to_date("order_purchase_timestamp").alias("dt"))
    .union(orders.select(to_date("order_approved_at").alias("dt")))
    .union(orders.select(to_date("order_delivered_carrier_date").alias("dt")))
    .union(orders.select(to_date("order_delivered_customer_date").alias("dt")))
    .union(orders.select(to_date("order_estimated_delivery_date").alias("dt")))
    .union(items.select(to_date("shipping_limit_date").alias("dt")))
    .union(reviews.select(to_date("review_creation_date").alias("dt")))
    .union(reviews.select(to_date("review_answer_timestamp").alias("dt")))
    .filter(col("dt").isNotNull())
    .distinct()
)

In [0]:
window = Window.orderBy("dt")

dim_date = (
    dates
    .select(
        row_number().over(window).alias("date_sk"),

        col("dt").alias("full_date"),

        dayofmonth("dt").alias("day"),

        month("dt").alias("month"),

        quarter("dt").alias("quarter"),

        year("dt").alias("year"),

        dayofweek("dt").alias("weekday")
    )
)

dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.dwh.dim_date")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

orders = spark.table("olist.stg.orders")
reviews = spark.table("olist.stg.order_reviews")
customers = spark.table("olist.dwh.dim_customers")
dates = spark.table("olist.dwh.dim_date")

window = Window.orderBy(col("o.order_id"))
fact_orders = (
    orders.alias("o")

    .join(
        reviews.alias("r"),
        col("o.order_id") == col("r.order_id"),
        "left"
    )

    .join(
        customers.alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left"
    )

    .join(
        dates.alias("d"),
        to_date(col("o.order_purchase_timestamp")) == col("d.full_date"),
        "left"
    )

    .select(

        row_number().over(window).alias("order_sk"),

        col("o.order_id"),

        col("c.customer_sk"),

        col("d.date_sk").alias("purchase_date_sk"),

        col("o.order_status"),

        col("o.is_approved"),

        col("o.is_shipped"),

        col("o.is_delivered"),

        col("r.review_score"),

        col("o.order_purchase_timestamp"),

        col("o.order_approved_at"),

        col("o.order_delivered_customer_date")

    )
)

fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.dwh.fact_orders")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
items = spark.table("olist.stg.order_items")

orders = spark.table("olist.dwh.fact_orders")

products = spark.table("olist.dwh.dim_products")

sellers = spark.table("olist.dwh.dim_sellers")

dates = spark.table("olist.dwh.dim_date")

window = Window.orderBy("order_id","order_item_id")

fact_order_items = (

    items.alias("i")

    .join(
        orders.alias("o"),
        "order_id",
        "left"
    )

    .join(
        products.alias("p"),
        col("i.product_id")==col("p.product_id"),
        "left"
    )

    .join(
        sellers.alias("s"),
        col("i.seller_id")==col("s.seller_id"),
        "left"
    )

    .join(
        dates.alias("d"),
        to_date(col("i.shipping_limit_date"))==col("d.full_date"),
        "left"
    )

    .select(

        row_number().over(window).alias("order_item_sk"),

        col("o.order_sk"),

        col("p.product_sk"),

        col("s.seller_sk"),

        col("d.date_sk").alias("shipping_date_sk"),

        col("i.order_item_id"),

        col("i.price"),

        col("i.freight_value"),

        col("i.shipping_limit_date")

    )

)

fact_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.dwh.fact_order_items")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
payments = spark.table("olist.stg.order_payments")

orders = spark.table("olist.dwh.fact_orders")

window = Window.orderBy("order_id","payment_sequential")

fact_payments = (

    payments.alias("p")

    .join(
        orders.alias("o"),
        "order_id",
        "left"
    )

    .select(

        row_number().over(window).alias("payment_sk"),

        col("o.order_sk"),

        col("p.payment_type"),

        col("p.payment_installments"),

        col("p.payment_value"),

        col("p.payment_sequential")

    )

)

fact_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("olist.dwh.fact_payments")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(spark.sql("SHOW TABLES IN olist.dwh"))

database,tableName,isTemporary
dwh,dim_customers,false
dwh,dim_date,false
dwh,dim_products,false
dwh,dim_sellers,false
dwh,fact_order_items,false
dwh,fact_orders,false
dwh,fact_payments,false


In [0]:
spark.table("olist.dwh.fact_orders").count()

99992